# H&M M5_B — 마지막 주 seed49

**현재 확정 명칭 M5_B = 적응형 M2 + M4(고객 내 평균1 CLV 가중 BPR).**
과거 M3+N/V 분리형 M4의 M5-B와 다릅니다.

- 새 학습은 **M5_B 한 번만** 합니다. 기존 M1·M2·M3·M4 seed49의 저장 지표를 재사용하며 재학습·재평가하지 않습니다.
- M2 표현 + **이진 그래프** + M4 손실을 하나의 optimizer로 공동 학습합니다. 별도 M3 가치그래프를 생성하지 않습니다.
- 기존 checkpoint로 초기화하거나 참조모형을 동결하지 않고 새 모델을 처음부터 공동 학습합니다.
- train 2018-09-20~2020-09-15 / test 2020-09-16~22, **300epoch 마지막 점 한 번 평가**. validation·holdout·조기종료 없음.
- ID64·2층·batch131072·lr0.0005·ID L2 0.001·uniform 음성1개, MIN_ITEM_INTER=1, 학습구간 구매쌍은 정답·후보에서 제외.
- CLV 입력창은 기존과 같은 학습 말미365일입니다. 계수·수식·epoch는 기존 확정 설정을 유지합니다.
- 이미 관찰한 test의 단일시드 추가 비교입니다. 통계적 유의성·CLV 귀속·새 독립 확증을 주장하지 않습니다.

런타임 → 런타임 유형 변경 → **L4 GPU**를 선택하고 위에서부터 실행하세요.
중단되면 같은 노트북·경로로 재실행합니다. 완료 결과는 건너뛰고 미완료 학습은 저장 epoch에서 재개합니다.
진행 중인 기존 H&M M5_C의 폴더와 별도 저장하므로 해당 실험을 변경하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 요청한 L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '828de460d197e1385e61b8c9efb973aaae857c95'
REPO = Path('/content/clv-hm-m5-followup-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 기존 결과 확인·입력 준비 (학습 없음)
Drive의 기존 M2 결과와 M4 seed49 내보내기 폴더에서 원본을 확인합니다. M4 파일에는 M1·M3 참조도 들어 있습니다.
원본 바이트와 입력·분할·특징·가중치가 다르면 **학습 전에 중단**하고 참조를 대신 학습하지 않습니다.

폴더를 옮겼거나 후속 실행으로 보고서가 갱신됐다면, 보관한 원본 ZIP의 Drive 경로를 아래 두 인수에 지정하세요:
`clv_m2_adaptive_hm_seed49_lastweek_results.zip`, `clv_hm_m4c_lastweek_seed49_results.zip`.
처음 실행한 뒤에는 참조·출력 경로를 유지하세요.

In [ ]:
import clv_m5_ab_adaptive_lastweek_seed49 as screen
cfg = screen.configure(dataset='hm', seed=49,
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_hm_seed49_lastweek_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_hm_m4c_lastweek_seed49_results.zip',
)
print('M2 참조:', cfg.m2_reference)
print('M1/M3/M4 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('재사용:', [screen.labels_for(cfg)[m] for m in prepared['references']['absolute'].model_id])
print('새 학습: M5_B(M2+M4) seed49 한 번, 300epoch')

## 2. M5_B 한 모형만 학습
학습 중 loss만 표시하고 test는 마지막300epoch에서 한 번만 평가합니다.
학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 같은 seed49의 다섯 모형 비교
M1·M2·M3·M4는 저장된 참조값, M5_B만 새 학습 결과입니다. **1시드 결과이며 10시드 평균이 아닙니다.**
핵심지표와 별도로 전체142지표·세그먼트·노출·진단이 CSV에 저장됩니다.
준비 셀 실행 후 이 셀은 저장된 완료값만 읽으며 추가 학습·test 평가를 하지 않습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
           'price_purchase_amount_weighted_hit@10', 'vndcg@10',
           'price_purchase_amount_weighted_hit@20', 'vndcg@20',
           'price_purchase_amount_weighted_hit@50', 'vndcg@50']
display(result['absolute'].set_index('model')[metrics].T.reindex(
    columns=['M1', 'M2', 'M3', 'M4', 'M5_B']))
display(result['comparison'].query("reference == 'M1'").pivot(
    index='metric', columns='model', values='relative_change_pct').reindex(metrics)
    if len(result['comparison']) else result['comparison'])
print('M1 대비 상대변화(%), 1시드 결과 — 유의성 주장 없음')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/hm_m5_b_adaptive_lastweek_seed49_results.zip')
files.download(output)